# Unit quality on synthetic units

Synthetic data only, so this runs anywhere `pip install jnwb` works. Five made-up units on an
eight-contact probe get the seven unit-quality measures, each with its unit and the cut-off the
caller chose. The notebook then draws the waveforms down the probe and shows the four outcomes
that `jnwb-qc` routes unit quality to: a measure executed, a missing input requested, a result
reported as not estimable, and a single-neuron claim declined.

A measure reads one unit and keeps or rejects none. Every number below is computed in this
notebook from the arrays it builds; none says anything about recorded neurons.

In [ ]:
import numpy as np
import pandas as pd
import jnwb
import jnwb.visual_qc as vqc

rng = np.random.default_rng(0)
fs = 30000.0                       # Hz, the sampling rate of the waveforms
n_samples = 48
t_ms = np.arange(n_samples) / fs * 1000.0
depths_um = 20.0 * np.arange(8)    # contact positions down the probe, in micrometres
duration_s = 600.0                 # recording length, in seconds

## 1. Build five units

Each unit has a mean waveform `(n_channels, n_samples)` in microvolts, whose amplitude falls with
distance from the unit's depth, and a spike train in seconds.

| Unit | Built to |
|---|---|
| `compact` | large, falls off within a few contacts; refractory period respected |
| `broad` | medium, spread over every contact; refractory period respected |
| `small` | amplitude of 18 uV |
| `contaminated` | independent Poisson train, no refractory period |
| `late` | fires only in the first 200 s of the recording |

In [ ]:
def template(amplitude_uv, centre_um, length_um):
    shape = -np.exp(-0.5 * ((np.arange(n_samples) - 16) / 2.0) ** 2) \
            + 0.35 * np.exp(-0.5 * ((np.arange(n_samples) - 26) / 5.0) ** 2)
    shape = shape / (shape.max() - shape.min())
    gain = amplitude_uv * np.exp(-np.abs(depths_um - centre_um) / length_um)
    return gain[:, None] * shape[None, :]


def regular_train(rate_hz, span_s, refractory_s=0.002):
    # exponential intervals shifted by the refractory period: no interval is shorter
    n = int(rate_hz * span_s * 1.2)
    isi = refractory_s + rng.exponential(1.0 / rate_hz - refractory_s, size=n)
    times = np.cumsum(isi)
    return times[times < span_s]


def poisson_train(rate_hz, span_s):
    return np.sort(rng.uniform(0.0, span_s, size=rng.poisson(rate_hz * span_s)))


templates = {
    "compact": template(150.0, 60.0, 40.0),
    "broad": template(90.0, 80.0, 400.0),
    "small": template(18.0, 100.0, 40.0),
    "contaminated": template(120.0, 40.0, 60.0),
    "late": template(110.0, 120.0, 50.0),
}
trains = {
    "compact": regular_train(8.0, duration_s),
    "broad": regular_train(5.0, duration_s),
    "small": regular_train(6.0, duration_s),
    "contaminated": poisson_train(8.0, duration_s),
    "late": regular_train(6.0, 200.0),
}
{name: (templates[name].shape, len(trains[name])) for name in templates}

## 2. Waveform measures

`waveform_features` needs the sampling rate; here it is `fs`, defined above. The amplitude is in the
waveform's unit (uV) and the duration in ms. `waveform_snr` reads the unit's individual spikes on
one channel, so each unit gets 200 noisy copies of its peak channel, with noise of 12 uV.

The two flags compare against cut-offs this notebook chooses: a waveform is flat below 25 uV, and
sharp at or above 0.018 of the amplitude lost per micrometre. Neither flag has a published
cut-off, so each is reported with its value.

In [ ]:
FLAT_UV = 25.0           # chosen here; no published value
SHARP_PER_UM = 0.018     # chosen here, in 1/um; no published value
NOISE_UV = 12.0
n_spikes = 200

rows = []
for name, w in templates.items():
    feat = jnwb.waveform_features(w, fs)
    spikes = w[feat["peak_channel"]][None, :] + rng.normal(0.0, NOISE_UV, (n_spikes, n_samples))
    flat = jnwb.waveform_flatness(w, threshold=FLAT_UV)
    sharp = jnwb.spatial_derivative_sharpness(w, depths_um, threshold=SHARP_PER_UM)
    rows.append({
        "unit": name,
        "peak_channel": feat["peak_channel"],
        "amplitude_uV": feat["amplitude"],
        "trough_to_peak_ms": feat["trough_to_peak_ms"],
        "peak_trough_ratio": feat["peak_trough_ratio"],
        "snr": jnwb.waveform_snr(spikes),
        "is_flat (<25 uV)": flat["is_flat"],
        "sharpness_per_um": sharp["sharpness"],
        "is_sharp (>=0.018/um)": sharp["is_sharp"],
    })
waveform_table = pd.DataFrame(rows).set_index("unit")
waveform_table.round(3)

## 3. Spike-train measures

`presence_ratio` takes blocks in seconds (ten blocks of 60 s here). `refractory_contamination`
requires the recording duration and both periods; here the refractory period is 2 ms and the
censored period 0.5 ms. `isi_cv` is unitless.

In [ ]:
blocks = np.column_stack([np.arange(0.0, duration_s, 60.0), np.arange(60.0, duration_s + 1, 60.0)])

rows = []
for name, spk in trains.items():
    rc = jnwb.refractory_contamination(spk, duration_s=duration_s, refractory_ms=2.0, censored_ms=0.5)
    rows.append({
        "unit": name,
        "n_spikes": len(spk),
        "presence_ratio (10 blocks of 60 s)": jnwb.presence_ratio(spk, blocks),
        "isi_cv": jnwb.isi_cv(spk),
        "n_violations (<2 ms)": rc["n_violations"],
        "contamination": rc["contamination"],
    })
train_table = pd.DataFrame(rows).set_index("unit")
train_table.round(3)

## 4. Waveforms down the probe

`plot_unit_waveforms` with `channels="all"` draws every contact of each unit, so the spread of a
unit over the probe is visible. The values are drawn as given; `voltage_unit` only writes the unit
on each axis, so pass the unit of your data.

In [ ]:
figs = vqc.plot_unit_waveforms(
    list(templates), templates, channels="all", voltage_unit="uV", figsize=(12, 6)
)
figs[0]

## 5. The four outcomes

**Execute** when the inputs are at hand: every measure above ran on arrays this notebook built.

**Request** what a measure lacks. A spatial derivative needs two or more channels and their
positions; a waveform measure needs `fs`. With the peak channel alone, the measure refuses and
names what to ask for.

In [ ]:
peak = waveform_table.loc["compact", "peak_channel"]
one_channel = templates["compact"][peak:peak + 1]
try:
    jnwb.spatial_derivative_sharpness(one_channel, depths_um[peak:peak + 1], threshold=SHARP_PER_UM)
except ValueError as err:
    print("request the probe geometry:", err)

**Report not estimable** when the input cannot support the value. A NaN is reported as such, never
read as 0 or as a plausible number. Three cases: a train whose violations leave the contamination
equation with no real root, a single spike (no spread to estimate a noise level from), and a
train with under three spikes (no coefficient of variation of intervals).

In [ ]:
rc = jnwb.refractory_contamination(trains["contaminated"], duration_s=duration_s,
                                   refractory_ms=2.0, censored_ms=0.5)
single = trains["compact"][:1]
print("contaminated unit:", rc["contamination"], "|", rc["reason"])
print("snr of one spike:", jnwb.waveform_snr(templates["compact"][peak:peak + 1, :]))
print("isi_cv of two spikes:", jnwb.isi_cv(trains["compact"][:2]))

**Decline** a single-neuron claim. The table shows `compact` with a high SNR, an amplitude above the
flat cut-off and no violations, and the measures still cannot say that its spikes come from one
neuron: they describe a waveform and a train, and two neurons with similar waveforms would score the
same way. The cell below checks only what the measures do establish, under the cut-offs stated with
each line.

In [ ]:
u = "compact"
print(f"{u}: amplitude {waveform_table.loc[u, 'amplitude_uV']:.0f} uV (flat below {FLAT_UV:.0f} uV), "
      f"sharpness {waveform_table.loc[u, 'sharpness_per_um']:.3f} per um "
      f"(sharp at or above {SHARP_PER_UM} per um), "
      f"{int(train_table.loc[u, 'n_violations (<2 ms)'])} intervals under 2 ms.")
print("Single neuron? Declined: no measure here shows that a unit is a single neuron.")